In [ ]:
import yaml
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from src.estate import BuyingScenario, evaluate_buying, _get_mortgage_config
from src.tax import NLHomeTax2026
from src.mortgage import MortgageLoan, amortize_multi_loan
from src.utils import future_value_monthly_growth


In [ ]:
save_figs = False

In [ ]:


def list_scenario_files(directory='scenarios'):
    """List all YAML scenario files (excluding templates)."""
    files = [f for f in os.listdir(directory) 
             if f.endswith('.yaml') and not f.endswith('.template')]
    return sorted(files)


def load_config(filepath):
    """Load YAML configuration file."""
    with open(filepath, 'r') as f:
        return yaml.safe_load(f)


def parse_buying_scenario(config):
    """Parse config and create BuyingScenario with tax."""
    name = config.get('name', 'Unknown')
    
    # Create tax object
    tax = NLHomeTax2026(**config['tax']) if 'tax' in config else None
    
    # Parse buying config
    buying_config = config['buying'].copy()
    
    # Convert mortgage_loans dicts to MortgageLoan objects
    if 'mortgage_loans' in buying_config and buying_config['mortgage_loans']:
        buying_config['mortgage_loans'] = [
            MortgageLoan(**loan_dict) 
            for loan_dict in buying_config['mortgage_loans']
        ]
    
    # Create scenario
    scenario = BuyingScenario(**buying_config, tax=tax)
    
    return name, scenario


def evaluate_scenarios(scenarios):
    """Evaluate all scenarios and return results."""
    results = {}
    for name, scenario in scenarios.items():
        results[name] = evaluate_buying(scenario)
    return results


def plot_final_wealth(ax, results):
    """Plot 1: Final wealth comparison (bar chart)."""
    names = list(results.keys())
    wealth = [results[name]['wealth_end'] for name in names]
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(names)))
    bars = ax.bar(range(len(names)), wealth, color=colors)
    
    ax.set_xlabel('Scenario')
    ax.set_ylabel('Final Wealth (€)')
    ax.set_title('Final Wealth Comparison')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, wealth)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val/1000:.0f}k', ha='center', va='bottom', fontsize=7)


def plot_monthly_costs(ax, results):
    """Plot 2: Monthly costs over time (line plot)."""
    for name, result in results.items():
        months = range(len(result['monthly_net_cost']))
        ax.plot(months, result['monthly_net_cost'], label=name, linewidth=1.5)
    
    ax.set_xlabel('Month')
    ax.set_ylabel('Monthly Net Cost (€)')
    ax.set_title('Monthly Costs Over Time')
    ax.legend(fontsize=7, loc='best')
    ax.grid(alpha=0.3)


def plot_cost_breakdown(ax, results):
    """Plot 3: Total cost breakdown (stacked bar chart)."""
    names = list(results.keys())
    
    # Extract cost components
    down_payments = [results[name]['down_payment'] for name in names]
    one_off = [results[name]['one_off_costs'] + results[name]['renovation_costs_once'] 
               for name in names]
    
    # Calculate monthly recurring (VvE + utilities)
    monthly_recurring = []
    for name in names:
        result = results[name]
        months = result['months']
        # Estimate from total spent minus mortgage and one-off
        total_mortgage = result['mortgage_interest_paid'] + result['mortgage_principal_paid']
        total_one_off = result['down_payment'] + result['one_off_costs'] + result['renovation_costs_once']
        recurring = result['total_spent'] - total_mortgage - total_one_off
        monthly_recurring.append(recurring)
    
    interest = [results[name]['mortgage_interest_paid'] for name in names]
    principal = [results[name]['mortgage_principal_paid'] for name in names]
    
    # Create stacked bar chart
    x = np.arange(len(names))
    width = 0.6
    
    ax.bar(x, down_payments, width, label='Down Payment', color='#1f77b4')
    ax.bar(x, one_off, width, bottom=down_payments, label='One-off Costs', color='#ff7f0e')
    
    bottom = np.array(down_payments, dtype=float) + np.array(one_off, dtype=float)
    ax.bar(x, monthly_recurring, width, bottom=bottom, label='Monthly Recurring', color='#2ca02c')
    
    bottom = bottom + np.array(monthly_recurring, dtype=float)
    ax.bar(x, interest, width, bottom=bottom, label='Mortgage Interest', color='#d62728')
    
    bottom = bottom + np.array(interest, dtype=float)
    ax.bar(x, principal, width, bottom=bottom, label='Mortgage Principal', color='#9467bd')

    # Annotate bars with total money spent
    for i, name in enumerate(names):
        total = results[name]['total_spent']
        ax.text(
            x[i], 
            down_payments[i] + one_off[i] + monthly_recurring[i] + interest[i] + principal[i] + 2000,  # offset above bar
            f"€{total:,.0f}", 
            ha='center', va='bottom', fontsize=7, color='black'
        )
    
    ax.set_xlabel('Scenario')
    ax.set_ylabel('Total Cost (€)')
    ax.set_title('Total Cost Breakdown')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.legend(fontsize=7, loc='lower left')
    ax.grid(axis='y', alpha=0.3)


def plot_equity_buildup(ax, results, scenarios):
    """Plot 4: Equity buildup over time (line plot)."""
    for name, result in results.items():
        scenario = scenarios[name]
        months = result['months']
        
        # Calculate home value and equity over time
        equity_over_time = []
        for m in range(months):
            home_value = future_value_monthly_growth(
                scenario.purchase_price, 
                scenario.annual_value_growth, 
                m
            )
            # Approximate remaining debt (simplified - actual would need mortgage schedule)
            # Use linear approximation based on final remaining balance
            remaining_debt = result['mortgage_remaining_balance'] + \
                           (result['mortgage_principal_paid'] * (months - m) / months)
            
            equity = home_value - remaining_debt
            equity_over_time.append(equity)
        
        ax.plot(range(months), equity_over_time, label=name, linewidth=1.5)
    
    ax.set_xlabel('Month')
    ax.set_ylabel('Home Equity (€)')
    ax.set_title('Equity Buildup Over Time')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)


def plot_interest_vs_principal(ax, results, scenarios):
    """Plot 5: Cumulative interest vs principal payments (stacked area)."""
    # Pick first 3 scenarios for clarity (or all if few)
    names = list(results.keys())[:3]

    line_styles = ['-', '--', '-.', ':']
    
    for i, name in enumerate(names):
        scenario = scenarios[name]
        result = results[name]
        
        # Get actual mortgage schedule with interest/principal breakdown
        loans = _get_mortgage_config(scenario)
        sched = amortize_multi_loan(
            loans=loans,
            horizon_months=scenario.living_months,
            rate_reset_values=None
        )
        
        # Get actual monthly interest and principal arrays
        interests = sched["interests"]
        principals = sched["principals"]
        
        # Calculate cumulative sums
        cum_interest = np.cumsum(interests)
        cum_principal = np.cumsum(principals)
        
        x = np.arange(len(interests))
        
        # Plot stacked areas
        ax.plot(x, cum_interest, alpha=0.4, label=f'{name} - Interest', linestyle=line_styles[0 % len(line_styles)], color=f'C{i}')
        ax.plot(x, cum_principal, alpha=0.4, label=f'{name} - Principal', linestyle=line_styles[1 % len(line_styles)], color=f'C{i}')
    
    ax.set_xlabel('Month')
    ax.set_ylabel('Cumulative Payments (€)')
    ax.set_title('Interest vs Principal Payments (First 3 Scenarios)')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.3)

In [ ]:

files = list_scenario_files('scenarios')
print(f"\nFound {len(files)} scenario files")
   

In [ ]:
 
# 2. Load and parse buying scenarios
scenarios = {}
for file in files:
    try:
        config = load_config(f'scenarios/{file}')
        if 'buying' in config:
            name, scenario = parse_buying_scenario(config)
            scenarios[name] = scenario
            print(f"  ✓ Loaded: {name} ({file})")
    except Exception as e:
        print(f"  ✗ Failed to load {file}: {e}")

print(f"\n✓ Successfully loaded {len(scenarios)} buying scenarios")


In [ ]:
filters = [
    " / ",
    "/ 30y",
    "ASN"
]
scenarios = {
    key: value for key, value in scenarios.items() if all(f in key for f in filters)
}
list(scenarios.keys())

In [ ]:

# 3. Evaluate all scenarios
print("\nEvaluating scenarios...")
results = evaluate_scenarios(scenarios)
print("✓ Evaluation complete")


In [ ]:

# 4. Create visualizations
print("\nGenerating plots...")
fig = plt.figure(figsize=(18, 10))

# Create 2x3 grid
ax1 = plt.subplot(2, 3, 1)
ax2 = plt.subplot(2, 3, 2)
ax3 = plt.subplot(2, 3, 3)
ax4 = plt.subplot(2, 3, 4)
ax5 = plt.subplot(2, 3, 5)

plot_final_wealth(ax1, results)
plot_monthly_costs(ax2, results)
plot_cost_breakdown(ax3, results)
plot_equity_buildup(ax4, results, scenarios)
plot_interest_vs_principal(ax5, results, scenarios)

fig.suptitle('Buy Scenarios Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()

# Save output
if save_figs:
    output_path = 'outputs/nb04_buy_scenarios_comparison.png'
    os.makedirs('outputs', exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved plot to: {output_path}")

plt.show()
